In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from keras.utils import to_categorical
from tensorflow.keras.layers import Dense, Dropout, Flatten, Dense, Dropout, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor = "val_loss", mode = "min", patience = 5)

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
EPOCHS = 1000


In [ ]:
root_path = "/usr/share/MSI25/data_images/"

train_dataset = tf.keras.utils.image_dataset_from_directory(
    root_path,
    label_mode="categorical",
    class_names=["smile", "non_smile"],
    batch_size=32,
    image_size=(64, 64),
    validation_split=0.2,
    subset="training",
    seed=123,
    shuffle=True
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    root_path,
    label_mode="categorical",
    class_names=["smile", "non_smile"],
    batch_size=32,
    image_size=(64, 64),
    validation_split=0.2,
    subset="validation",
    seed=123,
    shuffle=True
)

# odpowiednik rescale=1./255 - piksele z zakresu 0-255 na 0-1
train_dataset = train_dataset.map(lambda x, y: (x / 255.0, y))
validation_dataset = validation_dataset.map(lambda x, y: (x / 255.0, y))

In [ ]:
vgg_model = Sequential()

pretrained_model= tf.keras.applications.VGG16(include_top=False,
                   input_shape=(64,64,3),
                   weights='imagenet') 
for layer in pretrained_model.layers: #kluczowe
        layer.trainable=False

vgg_model.add(pretrained_model)

vgg_model.add(Flatten())
vgg_model.add(Dense(2, activation='softmax'))
vgg_model.summary()

In [ ]:
vgg_model.compile(optimizer="adam", loss='categorical_crossentropy',metrics=['accuracy'])

history = vgg_model.fit(train_dataset, validation_data=validation_dataset, epochs=EPOCHS, callbacks = [es])

In [ ]:
hist = pd.DataFrame(history.history)
plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.plot(hist.loss, label = "loss")
plt.plot(hist.val_loss, label = "val_loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()

plt.subplot(1,2,2)
plt.plot(hist.accuracy, label = "acc")
plt.plot(hist.val_accuracy, label = "val_acc")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()

In [ ]:
# Znajdz w internecie listę dostępnych modeli i sprawdź ich skuteczność